In [0]:
# =============================================================================
# NB14 — Within-paper results: bias controls before physical interpretation
# =============================================================================
# Context (NB13, strict blocks = same paper + same reported conditions):
#   XGBoost COMPACT pair accuracy 0.712 (permutation p = 0.016; null 0.50);
#   within-block OLS: measured Eg −0.354 log10 per SD (p_bonf = 0.050).
#
# Threat to validity — paper design / publication bias:
#   Papers are built around a "novel" material (doped, composite) that beats a
#   reference (pristine host), and the modification often narrows Eg. A model
#   could reach ~0.7 pair accuracy by learning "modified wins" rather than
#   transferable physics.
#
# This notebook:
#   A. Diagnoses WHAT is being compared inside blocks (hosts vs hosts, or
#      pristine vs modified), using Compound_original, dopant_paper,
#      secondary_phase_paper, sample_in_paper.
#   B. Training-free heuristic baselines on the same pairs:
#        modified_wins   — any dopant / secondary phase beats pristine
#        lower_Eg_wins   — smaller measured band gap wins
#        more_elements   — more distinct elements in base_compound wins
#   C. Out-of-fold XGBoost (COMPACT) predictions, 5×5 GroupKFold by DOI, and
#      pair accuracy STRATIFIED by pair type. Decisive question: does the
#      model still rank correctly among pairs where "modified vs pristine"
#      cannot help (both pristine or both modified)?  Permutation test on
#      that subset.
#   D. Ridge re-run with the scaling fix (drop features that are ~constant
#      within blocks in the training fold; clip scaled values) — NB13 Ridge
#      R² values (−0.5 to −92) were a numerical artefact.
#
# Inputs:  master, X_material, X_exp_v2
# Outputs: ml_perovskites.within_pairs_nb14, /tmp/nb14_pairs.png
# Rules:   DOI-grouped CV; imputation/scaling fitted on training rows; SEED=42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import re, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model  import RidgeCV
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS = 42, "ml_perovskites", 5, 5
PAIR_MIN, N_PERM = 0.2, 60

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Load and build strict blocks (identical definition to NB13)
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp_v2").toPandas()

def is_use_h2(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
master["bandgap_eV_meas"]  = pd.to_numeric(master["bandgap_eV"], errors="coerce")
df_base = master[is_use_h2(master["use_h2"]) & master["y_log10_1p_final"].notna() &
                 master["doi_key"].notna() & master["base_compound"].notna()].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

for t in [df_base, X_mat_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()
for c in X_mat_raw.columns:
    if c != "source_row_id":
        X_mat_raw[c] = pd.to_numeric(X_mat_raw[c], errors="coerce")
CAT_EXP = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type", "prep_class"]
for c in X_exp_raw.columns:
    if c not in ["source_row_id"] + CAT_EXP:
        X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")

PROV = [c for c in ["Compound_original", "dopant_paper", "dopant_fraction_paper",
                    "secondary_phase_paper", "sample_in_paper"] if c in df_base.columns]
COMPACT = [c for c in ["ECB_butler", "EVB_butler", "chi_mulliken", "goldschmidt_t",
                       "octahedral_mu", "Ra_Rb_ratio", "sigma_r_A", "sigma_r_B",
                       "wav_gs_bandgap", "wav_en_pauling", "wav_ionic_radius_shannon_vi",
                       "wav_num_d_valence", "wav_mendeleev_number", "wav_first_ion_en",
                       "wav_electron_affinity"] if c in X_mat_raw.columns]
BLOCK_COND = [c for c in ["run_type", "scavenger_class", "co_catalyst_type",
                          "co_catalyst_loading", "cutoff_nm_final", "lamp_power_W",
                          "catalyst_mass_g", "volume_mL", "organic_vol_fraction",
                          "T_reaction_C"] if c in X_exp_raw.columns]

df = (df_base[["source_row_id", "doi_key", "base_compound", "y", "bandgap_eV_meas"] + PROV]
      .merge(X_mat_raw[["source_row_id"] + COMPACT], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + BLOCK_COND], on="source_row_id", how="left"))
df["base_compound"] = df["base_compound"].astype(str).str.strip()

def cond_str(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.map(lambda v: "na" if pd.isna(v) else f"{v:.4g}")
    return s.astype(str).fillna("na")

df["block"] = df["doi_key"].astype(str)
for c in BLOCK_COND:
    df["block"] = df["block"] + "|" + cond_str(df[c])

EMPTY = {"", "nan", "none", "null", "-", "n/a", "na", "undoped", "pristine", "pure", "0"}
def present(s):
    s = s.where(s.notna(), "")          # NaN/None -> empty (pandas 1.x-3.x safe)
    return ~s.astype(str).str.strip().str.lower().isin(EMPTY)

df["is_modified_row"] = False
for c in ["dopant_paper", "secondary_phase_paper"]:
    if c in df.columns:
        df["is_modified_row"] |= present(df[c])

# ---------------------------------------------------------------------------
# A. What is compared inside strict blocks?
# ---------------------------------------------------------------------------
print("=== A. Provenance columns (top values) ===")
for c in PROV:
    vc = df[c].astype(str).str.strip().value_counts()
    print(f"\n{c}: {df[c].notna().sum()} non-null, {vc.size} distinct | top: "
          f"{vc.head(6).to_dict()}")

n_var = df.groupby(["block", "base_compound"])["Compound_original"].nunique() \
    if "Compound_original" in df.columns else None
if n_var is not None:
    print(f"\nRows merged per (block, base_compound): Compound_original variants "
          f"— {(n_var > 1).sum()} of {len(n_var)} groups merge ≥2 distinct original "
          f"names (e.g. doped/undoped versions of one host).")
    ex = n_var[n_var > 1].head(5).index
    for b, comp in ex:
        names = df[(df["block"] == b) & (df["base_compound"] == comp)]["Compound_original"].unique()
        print(f"   {comp:18s} ← {list(names)[:6]}")

def n_elements(formula):
    return len(set(re.findall(r"[A-Z][a-z]?", str(formula))))

def demean(v, blocks):
    v = pd.DataFrame(v)
    return (v - v.groupby(np.asarray(blocks)).transform("mean")).values

def heuristic_acc(score, subset):
    """Accuracy over pairs where the heuristic is decided (no tie / NaN)."""
    d = score[subset["j"].values] - score[subset["i"].values]
    ok = ~np.isnan(d) & (d != 0)
    if ok.sum() == 0:
        return np.nan, 0.0
    return np.mean(np.sign(d[ok]) == np.sign(subset["dy"].values[ok])), ok.mean()

def pair_acc_from_pred(pred, sub):
    d = pred[sub["j"].values] - pred[sub["i"].values]
    s = np.where(d == 0, 0.5, (np.sign(d) == np.sign(sub["dy"].values)).astype(float))
    return s.mean()

FEATS = ["eg"] + COMPACT   # measured Eg + compact descriptors (as NB13 COMPACT)
TYPES = ["all", "mixed", "both_pristine", "both_modified", "non_mixed",
         "non_mixed_diff_host", "equal_complexity"]
# non_mixed_diff_host : non-mixed pairs of DIFFERENT hosts (compositional
#                       descriptors are computed per host, so same-host pairs
#                       are indistinguishable except for measured Eg)
# equal_complexity    : non_mixed_diff_host AND same number of elements —
#                       neutralises both "modified wins" and "more complex
#                       composition wins" (e.g. SrTiO3 vs BaTiO3 vs CaTiO3)
PERM_SUBSETS = ["non_mixed", "non_mixed_diff_host", "equal_complexity"]

def subset(pairs, t):
    if t == "all":
        return pairs
    nm = pairs["type"] != "mixed"
    if t == "non_mixed":
        return pairs[nm]
    if t == "non_mixed_diff_host":
        return pairs[nm & ~pairs["same_host"]]
    if t == "equal_complexity":
        return pairs[nm & ~pairs["same_host"] & pairs["same_nel"]]
    return pairs[pairs["type"] == t]

def build(key):
    """Aggregate per (block, key); keep blocks with >= 2 distinct materials."""
    agg = (df.groupby(["block", key])
             .agg(doi_key=("doi_key", "first"), y=("y", "mean"),
                  eg=("bandgap_eV_meas", "mean"),
                  modified=("is_modified_row", "max"),
                  host=("base_compound", "first"),
                  **{c: (c, "mean") for c in COMPACT})
             .reset_index().rename(columns={key: "material"}))
    agg["n_el"] = agg["host"].map(n_elements)
    agg = agg[agg.groupby("block")["material"].transform("nunique") >= 2].reset_index(drop=True)
    pairs = []
    for b, idx in agg.groupby("block").indices.items():
        for a in range(len(idx)):
            for c in range(a + 1, len(idx)):
                i, j = idx[a], idx[c]
                dy = agg.at[j, "y"] - agg.at[i, "y"]
                if abs(dy) < PAIR_MIN:
                    continue
                mi, mj = bool(agg.at[i, "modified"]), bool(agg.at[j, "modified"])
                ptype = "mixed" if mi != mj else ("both_modified" if mi else "both_pristine")
                pairs.append({"block": b, "i": i, "j": j, "dy": dy, "type": ptype,
                              "same_host": agg.at[i, "host"] == agg.at[j, "host"],
                              "same_nel": agg.at[i, "n_el"] == agg.at[j, "n_el"]})
    return agg, pd.DataFrame(pairs)

def oof_predict(agg, yvals, rep, model_name):
    dois = np.array(sorted(agg["doi_key"].unique()))
    perm = np.random.default_rng(SEED + rep).permutation(dois)
    fold_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    f = agg["doi_key"].map(fold_of).values
    pred = np.full(len(agg), np.nan)
    for k in range(N_FOLDS):
        tr, va = np.where(f != k)[0], np.where(f == k)[0]
        if len(va) == 0:
            continue
        imp = SimpleImputer(strategy="median")
        Xtr = imp.fit_transform(agg.iloc[tr][FEATS].values.astype(float))
        Xva = imp.transform(agg.iloc[va][FEATS].values.astype(float))
        btr, bva = agg.iloc[tr]["block"].values, agg.iloc[va]["block"].values
        Xtr_w, Xva_w = demean(Xtr, btr), demean(Xva, bva)
        ytr = demean(yvals[tr], btr).ravel()
        if model_name == "XGBoost":
            m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                 subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                                 reg_lambda=2.0, tree_method="hist",
                                 random_state=SEED, verbosity=0)
            m.fit(Xtr_w, ytr)
            p = m.predict(Xva_w)
        else:
            # Ridge with scaling fix: drop features ~constant within training blocks
            raw_sd, within_sd = Xtr.std(axis=0), Xtr_w.std(axis=0)
            keep = (within_sd > 1e-9) & (within_sd > 0.05 * np.where(raw_sd > 0, raw_sd, 1))
            if keep.sum() == 0:
                pred[va] = 0.0
                continue
            sc = StandardScaler().fit(Xtr_w[:, keep])
            m = RidgeCV(alphas=np.logspace(-1, 4, 21)).fit(sc.transform(Xtr_w[:, keep]), ytr)
            p = m.predict(np.clip(sc.transform(Xva_w[:, keep]), -5, 5))
        pred[va] = demean(p, bva).ravel()
    return pred

HAS_ORIG = "Compound_original" in df.columns
KEYS = ["base_compound"] + (["Compound_original"] if HAS_ORIG else [])
rows = []
for key in KEYS:
    agg, pairs = build(key)
    y = agg["y"].values
    print(f"\n{'=' * 78}\nGROUPING KEY: {key}\n{'=' * 78}")
    print(f"Strict blocks: {agg['block'].nunique()} | DOIs: {agg['doi_key'].nunique()} | "
          f"materials: {len(agg)} | flagged modified: {int(agg['modified'].sum())}")
    if len(pairs) == 0:
        print("  no pairs — skipped"); continue
    print(f"Pairs with |Dy| >= {PAIR_MIN}: {len(pairs)} | by type: "
          f"{pairs['type'].value_counts().to_dict()} | same host: {int(pairs['same_host'].sum())}")

    print("\n-- B. Training-free heuristics: accuracy over decided pairs (coverage) --")
    heur = {"modified_wins": agg["modified"].astype(float).values,
            "lower_Eg_wins": -agg["eg"].values,
            "more_elements": agg["n_el"].astype(float).values}
    for name, sc in heur.items():
        parts = []
        for t in TYPES:
            sub = subset(pairs, t)
            if len(sub):
                acc, cov = heuristic_acc(sc, sub)
                parts.append(f"{t}: {acc:.3f} ({cov:.0%} of {len(sub)})")
                rows.append({"key": key, "method": f"heur_{name}", "pair_type": t,
                             "pair_acc": acc, "coverage": cov, "n_pairs": len(sub)})
        print(f"  {name:14s} | " + " | ".join(parts))

    print("\n-- C/D. Out-of-fold models (mean of 5 repeats) --")
    for mname in ["XGBoost", "Ridge_fixed"]:
        accs = {t: [] for t in TYPES}
        for rep in range(N_REPEATS):
            pred = oof_predict(agg, y, rep, "XGBoost" if mname == "XGBoost" else "Ridge")
            for t in TYPES:
                sub = subset(pairs, t)
                if len(sub):
                    accs[t].append(pair_acc_from_pred(pred, sub))
        for t, v in accs.items():
            if v:
                rows.append({"key": key, "method": mname, "pair_type": t,
                             "pair_acc": np.mean(v), "coverage": 1.0,
                             "n_pairs": len(subset(pairs, t))})
        print(f"  {mname:12s} | " + " | ".join(f"{t}: {np.mean(v):.3f}"
                                                for t, v in accs.items() if v))

    # Permutation test on pairs the "modified wins" heuristic cannot explain.
    # The model is retrained on y permuted within blocks; its predictions are
    # scored against the ORIGINAL pairs, so the null is "a model that learned
    # nothing about the true ranking".
    # The same permuted models are scored on every subset (no extra fits).
    subs = {t: subset(pairs, t) for t in PERM_SUBSETS}
    subs = {t: s for t, s in subs.items() if len(s) >= 20}
    for t in PERM_SUBSETS:
        if t not in subs:
            print(f"  [{t}] only {len(subset(pairs, t))} pairs — too few for a test.")
    print(f"\n-- Permutation test, XGBoost (repeat 0, {N_PERM} perms) --")
    if subs:
        pred0 = oof_predict(agg, y, 0, "XGBoost")
        obs = {t: pair_acc_from_pred(pred0, s) for t, s in subs.items()}
        rng = np.random.default_rng(SEED + 2000)
        bidx = list(agg.groupby("block").indices.values())
        null = {t: [] for t in subs}
        for _ in range(N_PERM):
            yp = y.copy()
            for idx in bidx:
                yp[idx] = y[rng.permutation(idx)]
            pp = oof_predict(agg, yp, 0, "XGBoost")
            for t, s in subs.items():
                null[t].append(pair_acc_from_pred(pp, s))
        for t, s in subs.items():
            nl = np.array(null[t])
            p = (1 + np.sum(nl >= obs[t])) / (1 + N_PERM)
            print(f"  {t:20s} observed {obs[t]:.3f} | null mean {nl.mean():.3f}, p95 "
                  f"{np.percentile(nl, 95):.3f} | p = {p:.3f} | n_pairs = {len(s)} | "
                  f"blocks = {s['block'].nunique()}")
            rows.append({"key": key, "method": f"XGBoost_perm_p_{t}", "pair_type": t,
                         "pair_acc": obs[t], "coverage": p, "n_pairs": len(s)})

res = pd.DataFrame(rows)

# ---------------------------------------------------------------------------
# Plot + write
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(KEYS), figsize=(7 * len(KEYS), 4), squeeze=False)
for ax, key in zip(axes[0], KEYS):
    r = res[res["key"] == key]
    for i, mname in enumerate(["XGBoost", "Ridge_fixed", "heur_modified_wins", "heur_lower_Eg_wins"]):
        s = r[r["method"] == mname].set_index("pair_type").reindex(TYPES)
        ax.bar(np.arange(len(TYPES)) + (i - 1.5) * 0.2, s["pair_acc"], 0.2, label=mname)
    ax.axhline(0.5, color="black", lw=0.8)
    ax.set_xticks(range(len(TYPES))); ax.set_xticklabels(TYPES, fontsize=8)
    ax.set_title(f"grouping: {key}"); ax.set_ylabel("Pair accuracy")
axes[0][0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("/tmp/nb14_pairs.png", dpi=120); plt.close()
print("\nSaved: /tmp/nb14_pairs.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.within_pairs_nb14")
print(f"✓ Written: {SCHEMA}.within_pairs_nb14 ({len(res)} rows)")
print("\n=== NB14 COMPLETE ===")